In [ ]:
import pandas as pd
from feat import Detector
from pathlib import Path

In [ ]:
result_path = "results/fig4_photos_pyfeat_paper.csv"

img_root = Path("../database/final_database")

image_paths = sorted(
    [p for p in img_root.rglob("*") if p.suffix.lower() == ".jpg"]
)
print(f"images found: {len(image_paths)}")

In [ ]:
detector = Detector()

rows = []

for image in image_paths:
    results = detector.detect_image(str(image))

    try:
        # select the face with the largest area
        areas = (
            results["FaceRectWidth"]
            * results["FaceRectHeight"]
        )
        largest_face_idx = areas.idxmax()
        largest_face = results.loc[largest_face_idx]

        row = dict(largest_face)

        # add information
        row["image_path"] = str(image)
        row["label"] = image.parent.name
        row["image_name"] = image.name

        rows.append(row)

    except Exception as e:
        print(f"error in {image}: {e}")


# save data
au_df = pd.DataFrame(rows)
meta_cols = ["image_path", "label", "image_name"]
other_cols = [c for c in au_df.columns if c not in meta_cols]
au_df = au_df[meta_cols + other_cols]
au_df.to_csv(result_path, index=False, encoding="utf-8-sig")
print(f"saved: {result_path}")